In [1]:
from utils.orthanc_utils import *
from utils.db_utils import *
from utils.plot_utils import *
from utils.mri_sorter import MRI_Sorter
from utils.model_utils import *

DB_PATH = "image_clasp_db.json"
ORTHANC = "http://localhost:8042"
METRICS_PATH = 'clasp_metrics.csv'
AUTH = ("orthanc","orthanc")

SESSION = requests.Session()
SESSION.auth = AUTH
SESSION.trust_env = False
studies = fetch_db_studies()
study = studies[0]
mri_sorter = MRI_Sorter(study)

In [2]:

dicom_info_2D = mri_sorter.dicom_info.loc[mri_sorter.dicom_info['Dimension'] == 2]

sort_dict_2D = {}

for series_uid, series_df in dicom_info_2D.groupby('SeriesUID'):
    # Initialize the dictionary for each series
    sort_dict_2D[series_uid] = {
        'Description': series_df.iloc[0]['SeriesDescription'],
        'N': len(series_df),
        'Dimension': 2
    }

# Initialize lists to store results
series_df_list = []

# Process unique ImageOrientationPatients
for unique_or in dicom_info_2D['ImageOrientationPatient'].dropna().unique():
    # Filter and sort DataFrame
    unique_df = dicom_info_2D[dicom_info_2D['ImageOrientationPatient'] == unique_or]
    unique_df = unique_df.set_index(['PixelSpacing', 'SliceThickness']).sort_index()

    # Group by unique indices (PixelSpacing, SliceThickness)
    for unique_idx, series_df in unique_df.groupby(level=[0, 1]):
        # Group by image shape
        for unique_imshape, shape_group in series_df.groupby('ImageShape'):
            # Filter for series with more than one entry
            series_df_list.append(series_df)

stack_df_list = [df for df in series_df_list if df['SliceLocation'].nunique() > 1]
single_df_list = [df for df in series_df_list if df['SliceLocation'].nunique() == 1]

static_stack_list = [
            df
            for df in stack_df_list
            if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() == 1
        ]

cine_stack_list = [
            df
            for df in stack_df_list
            if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() > 1
        ]

cine_stack_list, ss_mh_list = mri_sorter.split_by_similar_triggertime(cine_stack_list)

static_single_list = [
            df
            for df in single_df_list
            if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() == 1
        ]

cine_single_list = [
            df
            for df in single_df_list
            if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() > 1 
        ]

In [9]:
stack_df_list[0]

SeriesUID  \
PixelSpacing SliceThickness                                                 
1.289062     16.0            0980fa03-681913d2-e18d1e51-f4483387-51b3a0b2   
             16.0            0980fa03-681913d2-e18d1e51-f4483387-51b3a0b2   
             16.0            0980fa03-681913d2-e18d1e51-f4483387-51b3a0b2   

                                     SeriesDescription  Dimension  \
PixelSpacing SliceThickness                                         
1.289062     16.0            Single_shot_localiser_Sax          2   
             16.0            Single_shot_localiser_Sax          2   
             16.0            Single_shot_localiser_Sax          2   

                             N_timesteps  ImageShape  venc     rr  \
PixelSpacing SliceThickness                                         
1.289062     16.0                      1  (256, 224)     0  927.0   
             16.0                      1  (256, 224)     0  927.0   
             16.0                      1  (256, 224)     0  927.0   

                                                                       ID  \
PixelSpacing SliceThickness                                                 
1.289062     16.0            36bbf8f1-406bc95d-d80b02f7-f78b52f5-ffba7c84   
             16.0            250f192b-321c293b-c4ec77b2-f4ae3978-c6c9fb50   
             16.0            50ca207a-e42902c7-76f0d334-accc5dba-1ff3fdf4   

                                                       ImageOrientationPatient  \
PixelSpacing SliceThickness                                                      
1.289062     16.0            (0.41524577512456, 0.84224201097871, -0.343801...   
             16.0            (0.41524577512456, 0.84224201097871, -0.343801...   
             16.0            (0.41524577512456, 0.84224201097871, -0.343801...   

                                                          ImagePositionPatient  \
PixelSpacing SliceThickness                                                      
1.289062     16.0            (12.604901060021, -55.701145594993, 172.508629...   
             16.0            (24.949143155968, -64.121815150657, 166.789192...   
             16.0            (37.293384298241, -72.542488521018, 161.069755...   

                             SliceLocation  InstanceNumber  
PixelSpacing SliceThickness                                 
1.289062     16.0               172.508629               1  
             16.0               166.789193               2  
             16.0               161.069756               3

In [3]:
cine_stack_list[0]

IndexError: list index out of range